# C2 · 캡스톤 2: 대기행렬 설계 (Capstone 2: Designing a Service System with Erlang C and M/G/c Simulation)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.3 (CTMC의 극한 거동), §4.5 (마르코프 대기행렬 M/M/1, M/M/s) · Ross 12e Ch. 8: §8.2 (비용 항등식·리틀), §8.3 (지수 모형, M/M/k), §8.5 (M/G/1: Pollaczek–Khinchine) · G&S 4e Ch. 11 (대기행렬) · Kiefer & Wolfowitz (1955), Trans. AMS 78 (GI/G/c 작업량 재귀; Asmussen, *Applied Probability and Queues* 2e, Ch. XII) |
| 재사용 노트북 | 03c (출생–사망 정상분포·Gillespie 시간평균), 03d (M/M/1·리틀·PASTA), 03e (Erlang C/B), 03r, 02a (경쟁하는 시계·지수 합), 00b (SE·반복 실험), 00c (기하 합의 지수분포) |
| 예상 소요 | 120분 |
| 상태 | draft |

## 프로젝트 브리프 — 합격 기준과 단계

**브리프.** 콜센터(또는 창구)를 설계한다. 도착은 시간당 $\lambda = 40$의 푸아송 과정, 서비스는 평균 5분($\mu = 12$/시간), 제공 부하(offered load)는 $a = \lambda/\mu = 10/3$ Erlang이다. SLA(service level agreement)는 **"1분 넘게 기다리는 고객 비율이 10% 이하"**, 즉 $P(W_q > 1\text{분}) \le 0.10$. 비용은 서버 1대당 시간당 30, 고객 대기 1시간당 60이다. 과제는 (i) Erlang C로 SLA를 만족하는 최소 서버 수 $c^*$를 정하고, (ii) 비용 곡선과 대조하고, (iii) 서비스가 지수분포가 아닐 때(결정적 $c_s^2 = 0$, 초지수 $c_s^2 = 4$) 같은 $c$가 여전히 SLA를 지키는지 M/G/c 시뮬레이션으로 검사하는 것이다.

**합격 기준 — 이 그림·숫자가 나오면 성공.**

1. Erlang C 표: $c = 5$에서 $P(W_q > 1\text{분}) \approx 0.234$(위반), $c = 6$에서 $\approx 0.087$(만족) → $c^* = 6$. $c = 6$에서 $P(\text{wait}) \approx 0.148$, $W_q \approx 0.278$분, $L \approx 3.52$.
2. 비용 곡선의 최소는 $c = 5$(약 189/h)이지만 SLA 실현가능 집합(feasible set) 안에서는 $c = 6$(약 191/h).
3. M/M/c 시뮬레이션(Kiefer–Wolfowitz 재귀, 200회 반복 × 20,000 고객)이 $c = 5, 6, 7$ 모두에서 $P(W_q > 1\text{분})$, $P(W_q > 0)$, $W_q$를 Erlang C와 4 SE 이내로 재현하고, Gillespie CTMC(03c)로 $L$과 $P(N \ge 6)$도 재현한다(PASTA).
4. M/G/1 검증: $\rho = 0.8$에서 $W_q$ = 10분(D) / 20분(M) / 50분(H2, $c_s^2 = 4$) — Pollaczek–Khinchine와 4 SE 이내.
5. 강건성 그림: $c = 6$에서 결정적 서비스는 $\approx 0.066$(여유), $c_s^2 = 4$ 초지수는 $\approx 0.10$으로 SLA 경계에 걸리거나 위반 → 7대 필요($\approx 0.034$).

**단계.**

| 단계 | 내용 | 도구 |
|---|---|---|
| 1 | Erlang C 표와 SLA 곡선 → $c^*$ | `mmc_metrics` + inline `erlang_c` |
| 2 | 비용 곡선 $TC(c)$와 제약 최적화 | 리틀의 법칙 |
| 3 | M/M/c를 두 시뮬레이터로 검증 (Gillespie CTMC, Kiefer–Wolfowitz) | `gillespie_paths`, inline `mgc_waiting_times` |
| 4 | M/G/1로 비지수 서비스 검증 (Pollaczek–Khinchine) | inline `service_sampler` |
| 5 | M/G/c 강건성: 같은 $c$, 다른 서비스 분포 | 반복 실험 + `mc_estimate` |
| Compare | 모든 검증 수치를 한 표로 | `compare_table`, `assert_close` |
| 6 | 설계 메모와 대기실 없는 대안 (Erlang B) | `birth_death_stationary` |
| 스트레치 | S1 시간 변동 도착, S2 이탈(Erlang A), S3 빠른 1대 vs 느린 2대 | — |

## 0. 준비 (Setup)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import (mmc_metrics, birth_death_generator, birth_death_stationary, stationary_ctmc,
                        is_generator, gillespie, gillespie_paths, time_average_occupation)
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("C2")
setup_plots()
LAM, MU = 40.0, 12.0            # 도착률, 서비스율 (시간당)
T_SLA, ALPHA = 1 / 60, 0.10     # SLA: P(Wq > 1분) <= 0.10
COST_SERVER, COST_WAIT = 30.0, 60.0
R = 200                                   # 독립 반복 횟수 (FAST에서도 유지: SE는 반복 수로 결정)
N_CUST = scale(20_000, minimum=2_000)     # 반복당 고객 수 (FAST: 2,000)
WARMUP = scale(2_000)                     # 버리는 워밍업 고객 수 (FAST: 200)
N_PATHS = scale(2_000, minimum=1_000)     # Gillespie 경로 수 (FAST: 1,000 — 4 SE 검사가 의미 있으려면)
T_LONG = 200.0 if fast() else 2000.0      # 긴 Gillespie 경로의 길이 (시간)
print(f"SEED={SEED}, fast={fast()}, R={R}, N_CUST={N_CUST}, WARMUP={WARMUP}, N_PATHS={N_PATHS}")

## 1. Recall — 재사용 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03r/03d) M/M/1에서 $L, W, L_q, W_q$의 닫힌 형식과 리틀의 법칙을 쓰라.

<details><summary>정답</summary>

$\rho = \lambda/\mu < 1$일 때 $L = \rho/(1-\rho)$, $W = 1/(\mu-\lambda)$, $L_q = \rho^2/(1-\rho)$, $W_q = \rho/(\mu-\lambda)$. 리틀: $L = \lambda W$, $L_q = \lambda W_q$, $L = L_q + \rho$.

</details>

**Q2.** (03r/03e) Erlang C와 Erlang B는 각각 어떤 시스템의 어떤 확률인가?

<details><summary>정답</summary>

Erlang C: M/M/c(무한 대기실)에서 도착 고객이 기다릴 확률 $P(N \ge c)$. Erlang B: M/M/c/c(대기실 없음)에서 도착 고객이 차단될 확률 $P(N = c)$. 둘 다 제공 부하 $a = \lambda/\mu$와 $c$의 함수다.

</details>

**Q3.** (03r/03c) 출생–사망 과정의 정상분포 곱 형태를 쓰고 M/M/c의 출생·사망률을 말하라.

<details><summary>정답</summary>

$\pi_n = \pi_0 \prod_{k=0}^{n-1} \lambda_k/\mu_{k+1}$, $\pi_0$는 정규화 상수. M/M/c: $\lambda_k = \lambda$, $\mu_k = \min(k, c)\,\mu$.

</details>

**Q4.** (03d) PASTA는 무엇을 말하며, 왜 M/M/c의 대기확률이 $\sum_{n \ge c} \pi_n$인가?

<details><summary>정답</summary>

푸아송 도착이 보는 시스템 상태의 분포는 시간평균 정상분포 $\pi$와 같다(도착 과정의 미래 증분이 현재 상태와 독립이므로). 따라서 도착 고객이 '모든 서버가 바쁜 상태'를 볼 확률 $= \sum_{n \ge c} \pi_n$ = Erlang C.

</details>

## 2. 이론 — 설계에 필요한 정리를 내 말로

### 2.1 정의

- **M/M/c**: 푸아송($\lambda$) 도착, i.i.d. 지수($\mu$) 서비스, 서버 $c$대, 무한 FCFS(first come, first served) 대기실. 제공 부하 $a = \lambda/\mu$ (단위 Erlang), 이용률(utilization) $\rho = a/c < 1$.
- **Erlang C**: $C(c, a) = \dfrac{a^c/(c!(1-\rho))}{\sum_{k=0}^{c-1} a^k/k! + a^c/(c!(1-\rho))} = P(\text{도착 고객이 기다림})$. spkit `mmc_metrics(lam, mu, c)`의 `p_wait`.
- **대기시간 SLA**: $P(W_q > t^*) \le \alpha$; 여기서 $t^* = 1$분 $= 1/60$시간, $\alpha = 0.10$. M/M/c에서 $P(W_q > t) = C(c, a)\,e^{-(c\mu - \lambda)t}$ (정리 2).
- **M/G/c**: 서비스시간 분포가 일반(평균 $1/\mu$, 제곱변동계수 $c_s^2 = \mathrm{Var}(S)/E[S]^2$). 결정적(D): $c_s^2 = 0$; 지수(M): 1; **균형 초지수** H2($c_s^2 = 4$): 확률 $p_1$로 Exp($\mu_1$), $1-p_1$로 Exp($\mu_2$), 균형 조건 $p_1/\mu_1 = (1-p_1)/\mu_2$에서 $p_1 = \tfrac12\big(1 + \sqrt{(c_s^2-1)/(c_s^2+1)}\big) \approx 0.8873$, $\mu_1 = 2p_1\mu \approx 21.30$, $\mu_2 = 2(1-p_1)\mu \approx 2.705$ (시간당). 로그정규(logn, $c_s^2 = 4$): $\sigma^2_{\log} = \ln(1 + c_s^2) = \ln 5$.
- **Kiefer–Wolfowitz 작업량 벡터**: 고객 $n$이 도착 직전에 보는 각 서버의 잔여 작업량을 오름차순으로 정렬한 $w_n \in \mathbb{R}^c$. FCFS 대기시간은 $W_n = w_n[0]$이고 $w_{n+1} = R\big((w_n + S_n e_1 - A_n \mathbf{1})^+\big)$, $A_n$ = 고객 $n$과 $n+1$의 도착 간격, $R$ = 정렬.
- **비용함수**: $TC(c) = c_s\, c + c_w\, \lambda\, W_q(c) = c_s c + c_w L_q(c)$ (리틀). 설계 문제: $\min_c TC(c)$ s.t. $P(W_q > t^*) \le \alpha$.
- **반복 실험(replication)**: 독립 시드로 $R$번 시뮬레이션해 각 반복의 통계량(예: 1분 초과 비율)을 **하나의 표본**으로 보고 `mc_estimate`한다. 고객 단위 표본 $\{W_n\}$은 자기상관이 강해 $\mathrm{sd}/\sqrt{n}$이 SE를 심하게 과소평가하기 때문이다(00b, 03d).

### 2.2 정리 1 — Erlang C: M/M/c 정상분포와 대기확률 (Durrett 3e §4.5, Ross 12e §8.3)

> $\rho = a/c < 1$이면
> $$\pi_n = \pi_0 \frac{a^n}{n!}\ (n \le c), \qquad \pi_n = \pi_0 \frac{a^c}{c!}\rho^{\,n-c}\ (n \ge c), \qquad \pi_0^{-1} = \sum_{k<c} \frac{a^k}{k!} + \frac{a^c}{c!(1-\rho)}.$$
> 도착 고객의 대기확률은 $C(c, a) = \sum_{n \ge c} \pi_n = \pi_0 \dfrac{a^c}{c!(1-\rho)}$, 그리고 $L_q = C\rho/(1-\rho)$, $W_q = L_q/\lambda$.

**가정이 왜 필요한가**
- **$\rho < 1$**: 기하급수 $\sum_{n \ge c} \rho^{n-c}$가 수렴해야 정규화할 수 있다. 아니면 정상분포가 없다 — 대기열이 폭발한다(E3).
- **푸아송 도착 + 지수 서비스**: 그래야 $N(t)$가 출생–사망 과정이 되어 곱 형태(03c)가 성립한다. 서비스가 일반 분포면 $N(t)$는 마르코프가 아니다. M/G/1은 그래도 평균 대기의 닫힌 형식(정리 4)이 있지만, M/G/c($c \ge 2$)의 대기시간 분포에는 일반적인 닫힌 형식이 없다(단계 5는 그래서 시뮬레이션이다).
- **PASTA**: 시간평균 $\pi$를 '도착 고객이 보는' 분포로 바꾸는 데 필요하다. 비푸아송 도착(E3의 D/M/c)에서는 도착 고객이 보는 분포가 $\pi$와 다르다.

<details><summary>증명</summary>

출생률 $\lambda_n = \lambda$, 사망률 $\mu_n = \min(n, c)\mu$인 출생–사망 과정의 곱 형태(03c)에서
$$\frac{\pi_n}{\pi_0} = \prod_{k=0}^{n-1} \frac{\lambda}{\min(k+1, c)\mu}.$$
$n \le c$이면 분모가 $1\cdot 2 \cdots n \cdot \mu^n = n!\,\mu^n$이므로 $\pi_n/\pi_0 = a^n/n!$. $n > c$이면 처음 $c$개 인자가 $a^c/c!$을 주고 나머지 $n-c$개 인자는 각각 $\lambda/(c\mu) = \rho$이므로 $\pi_n/\pi_0 = \frac{a^c}{c!}\rho^{n-c}$.

정규화: $\sum_{n \ge c} \frac{a^c}{c!}\rho^{n-c} = \frac{a^c}{c!}\sum_{j \ge 0}\rho^j = \frac{a^c}{c!(1-\rho)}$ ($\rho < 1$). 따라서 $\pi_0^{-1} = \sum_{k<c} a^k/k! + a^c/(c!(1-\rho))$.

대기확률: PASTA(03d 정리 3)에 의해 도착 고객이 보는 $N$은 $\pi$를 따르고, 모든 서버가 바쁜 상태는 $N \ge c$이므로 $P(\text{wait}) = \sum_{n \ge c}\pi_n = \pi_0\frac{a^c}{c!(1-\rho)} = C(c, a)$.

대기열 길이: $L_q = \sum_{n \ge c}(n-c)\pi_n = \pi_0\frac{a^c}{c!}\sum_{j \ge 0} j\rho^j = \pi_0\frac{a^c}{c!}\cdot\frac{\rho}{(1-\rho)^2} = C\cdot\frac{\rho}{1-\rho}$ (기하급수의 미분 $\sum j\rho^j = \rho/(1-\rho)^2$). 리틀의 법칙으로 $W_q = L_q/\lambda$. $\square$

</details>

본 문제($a = 10/3$, $c = 6$, $\rho = 5/9$)에서 이 공식은 $C \approx 0.1482$, $L_q \approx 0.1853$, $W_q \approx 0.278$분, $L = L_q + a \approx 3.519$를 준다 — 단계 1에서 코드로 계산한다.

### 2.3 정리 2 — M/M/c 대기시간 분포 (Ross 12e §8.3, G&S 4e Ch. 11)

> FCFS M/M/c, $\rho < 1$: $P(W_q > t) = C(c, a)\,e^{-(c\mu - \lambda)t}$. 즉 $W_q \mid W_q > 0 \sim \mathrm{Exp}(c\mu - \lambda)$이고 $E[W_q \mid W_q > 0] = 1/(c\mu - \lambda)$ ($c = 6$: $1/32$시간 $= 1.875$분).

**가정이 왜 필요한가**
- **FCFS**: 도착 고객 앞의 대기 인원이 '내가 기다려야 할 출발 횟수'가 되는 근거. 다른 규율(LIFO, 무작위)에서는 $W_q$의 분포가 다르다(평균은 같다).
- **지수 서비스**: 모든 서버가 바쁠 때 출발 과정이 율 $c\mu$의 푸아송이 되는 근거(무기억성 + 경쟁하는 시계, 02a).
- **PASTA**: 도착 고객이 보는 $N$이 $\pi$를 따른다는 것.

<details><summary>증명</summary>

도착 고객이 $N = n \ge c$를 볼 조건부 확률은 정리 1의 꼬리 형태에서 $P(N - c = j \mid N \ge c) = (1-\rho)\rho^j$, 즉 기하분포다. 앞에 $j$명이 대기 중이므로 이 고객은 $j+1$번의 출발을 기다려야 서비스를 시작한다. 모든 서버가 바쁜 동안 $c$개의 Exp($\mu$) 시계가 경쟁하므로 출발 간격은 i.i.d. Exp($c\mu$)이고(02a: $c$개 지수 시계의 최소는 Exp($c\mu$), 무기억성으로 매 출발 후 새로 시작), 대기 중인 동안 서버가 비는 일은 없다.

따라서 $W_q \mid W_q > 0$은 '기하($1-\rho$, 지지집합 $1, 2, \dots$)개의 i.i.d. Exp($c\mu$)의 합' $T$다. mgf로(00c):
$$E[e^{sT}] = \sum_{j \ge 0}(1-\rho)\rho^j\Big(\frac{c\mu}{c\mu - s}\Big)^{j+1} = \frac{(1-\rho)c\mu}{c\mu - s}\cdot\frac{1}{1 - \rho\frac{c\mu}{c\mu - s}} = \frac{c\mu(1-\rho)}{c\mu(1-\rho) - s},$$
이는 Exp($c\mu(1-\rho)$)의 mgf이고 $c\mu(1-\rho) = c\mu - \lambda$. 그러므로 $P(W_q > t) = P(W_q > 0)\,P(T > t) = C(c, a)\,e^{-(c\mu-\lambda)t}$. $\square$

</details>

### 2.4 정리 3 — 리틀의 법칙과 PASTA (Durrett 3e §4.3, Ross 12e §8.2; 03d에서 확인, 04a에서 증명)

> $L = \lambda W$, $L_q = \lambda W_q$ (서비스 규율·분포 무관). 푸아송 도착이 보는 상태 분포 = 시간평균 분포.

**가정이 왜 필요한가**
- **리틀**: 장기 도착률과 장기 평균의 존재(안정성)만 필요하다 — 재생–보상 정리(04a)로 증명한다.
- **PASTA**: 도착 과정이 푸아송이고 미래 증분이 현재 상태와 독립(lack of anticipation)이어야 한다.

*증명은 인용한다*: 03d에서 시뮬레이션으로 확인했고 04a에서 재생–보상으로 증명한다. 이 노트북에서는 단계 3에서 Gillespie **시간평균** $P(N \ge c)$와 고객평균 $P(W_q > 0)$이 같은 값($\approx 0.148$)을 주는 것으로 PASTA를 재확인한다.

### 2.5 정리 4 — Pollaczek–Khinchine: M/G/1 평균 대기 (Ross 12e §8.5, G&S 4e Ch. 11)

> M/G/1, $\rho = \lambda E[S] < 1$:
> $$W_q = \frac{\lambda E[S^2]}{2(1-\rho)} = \frac{\rho(1 + c_s^2)}{2\mu(1-\rho)}.$$

**가정이 왜 필요한가**
- **푸아송 도착**: 도착 고객이 보는 잔여 작업량이 시간평균과 같아야(PASTA) 아래 평균값 논법이 닫힌다.
- **$\rho < 1$**: 안정성.
- **FCFS는 평균에 영향 없다**(작업 보존): 서비스시간 정보를 쓰지 않는 비선점·작업 보존 규율(FCFS, LIFO, 무작위)은 $W_q$의 분포만 바꾸고 평균은 바꾸지 않는다. 서비스시간을 보고 고르는 규율(SPT)이나 선점 규율은 평균도 바꾼다.

*증명 스케치 (평균값 논법)*: 도착 고객의 대기 = (앞에 대기 중인 고객들의 서비스 합) + (서비스 중인 고객의 잔여 서비스). 기댓값을 취하면 $W_q = L_q E[S] + \rho\, E[S_{\mathrm{res}}]$, 여기서 서버가 바쁠 확률은 $\rho$이고 잔여 서비스의 기댓값은 검사 역설(04b 잔여수명)로 $E[S_{\mathrm{res}}] = E[S^2]/(2E[S])$. 리틀 $L_q = \lambda W_q$를 대입하면 $W_q(1 - \rho) = \lambda E[S^2]/2$. $E[S^2] = (1 + c_s^2)/\mu^2$를 쓰면 둘째 표현. 검증값($\lambda = 9.6$, $\mu = 12$, $\rho = 0.8$): D 10분, M 20분, H2($c_s^2 = 4$) 50분 — 즉 $1 : 2 : 5$. **$E[S^2]$만이 전부다**: 이것이 M/G/1의 특별한 점이고, M/G/c($c \ge 2$)에서는 성립하지 않는다(함정 4).

### 2.6 정리 5 — Kiefer–Wolfowitz 재귀: GI/G/c FCFS (Kiefer & Wolfowitz 1955; Asmussen Ch. XII)

> $w_{n+1} = R\big((w_n + S_n e_1 - A_n\mathbf{1})^+\big)$, $W_n = w_n[0]$. $\rho < 1$이면 $w_n$은 정상 극한을 가지며 고객 평균이 그 극한의 기댓값으로 수렴한다.

**가정이 왜 필요한가**
- **FCFS + 작업 보존**: 고객 $n$은 가장 먼저 비는 서버(작업량 최소 = 정렬 후 첫 성분)에 배정된다.
- **i.i.d. 도착 간격·서비스(GI/G)**: 재귀가 마르코프 사슬(랜덤 워크형)이 되어 안정성 이론이 적용된다.

*증명 스케치*: $w_n[0]$은 가장 빨리 비는 서버까지 남은 시간 = FCFS 대기시간. 고객 $n$이 그 서버에 자기 작업 $S_n$을 얹고($+S_n e_1$), 다음 도착까지 $A_n$이 흘러 모든 서버의 작업량이 $A_n$만큼 줄며($-A_n\mathbf{1}$, 0 아래로는 못 내려감) 순서가 바뀔 수 있으므로 다시 정렬한다. $c = 1$이면 Lindley 재귀 $W_{n+1} = (W_n + S_n - A_n)^+$(03d)다. 구현: 반복 실험 축 $R$을 벡터화하고 고객 축만 파이썬 루프로 돈다(20,000회 × 수십 μs ≈ 1초).

### 2.7 직관

Erlang C는 '모든 서버가 바쁜 시간의 비율'이고, 그 순간에 도착한 고객의 대기는 율 $c\mu - \lambda$의 지수분포다. 서버를 하나 더 늘리면 $c\mu - \lambda$가 커져 꼬리가 **급격히** 짧아진다($c = 5 \to 6$: $0.234 \to 0.087$). 비용 곡선은 완만한 U자라 최적점 근처에서 1대 차이는 몇 % 차이지만, **SLA는 절벽**이다. 그리고 지수 서비스 가정은 낙관적이지도 비관적이지도 않은 '중간'($c_s^2 = 1$)이다: 실제 서비스가 더 규칙적이면 여유가 생기고, 더 들쭉날쭉하면($c_s^2 = 4$) 같은 $c$로는 SLA가 깨진다. 이 두 방향을 **숫자로** 아는 것이 이 캡스톤의 목표다.

### 2.8 함정(traps)

1. 비용 최적 $c = 5$와 SLA 최소 $c = 6$은 **다른 질문의 답**이다. 제약 최적화에서는 실현가능 집합에서 최소를 찾는다.
2. 고객 단위 표본 $\{W_n\}$은 자기상관이 강해 $\mathrm{sd}/\sqrt{n}$이 SE를 심하게 과소평가한다. 반복 실험 평균(i.i.d.)으로 SE를 구하라(00b).
3. 워밍업을 버리지 않으면(빈 시스템 시작) 대기가 과소평가된다. M/M/6의 완화 시간은 고객 10명 규모지만 H2는 더 길다 — 2,000명(FAST 200명)을 버린다.
4. $c_s^2$만으로 M/G/c의 대기가 결정되지 않는다. 근사 $W_q \approx W_q^{M/M/c}(1 + c_s^2)/2$는 $c = 6$의 D에서 0.139분을 예측하지만 시뮬레이션은 약 0.16분, H2에서 0.695 vs 약 0.5분이다. M/G/1(P–K)에서만 정확히 $E[S^2]$가 전부다.
5. Erlang C 공식에 $\rho \ge 1$을 넣으면 음수·무한이 나온다(`mmc_metrics`는 예외를 낸다) — 공식이 아니라 가정이 깨진 것이다(E3).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

In [ ]:
predictions = {
    "c_star": None,            # λ=40/h, μ=12/h에서 P(Wq>1분) <= 0.10을 만족하는 최소 서버 수 c*는?
    "p_wait_cstar": None,      # 그 c*에서 Erlang C (도착 고객이 기다릴 확률)는?
    "wq_cstar_min": None,      # 그 c*에서 평균 대기 Wq (분)는?
    "cond_wait_cstar_min": None,  # 그 c*에서 '기다리는 고객만'의 평균 대기 E[Wq | Wq>0] (분)는?
    "c_cost_opt": None,        # SLA를 무시하고 TC(c) = 30c + 60·λ·Wq(c)만 최소화하면 c는?
    "p_exceed_det_c6": None,   # c=6, 서비스가 5분 고정(D)이면 P(Wq>1분)은? (지수 서비스보다 클지 작을지 먼저 생각)
    "p_exceed_h2_c6": None,    # c=6, 서비스가 초지수(H2, c_s²=4)이면 P(Wq>1분)은?
    "wq_mh21_min": None,       # λ=9.6/h, μ=12/h (ρ=0.8)인 M/G/1에서 H2(c_s²=4) 서비스의 평균 대기 Wq(분)는?
}

## 4. 단계 1 — Erlang C 표와 SLA 곡선

먼저 정리 1을 **직접 구현**해 spkit의 `mmc_metrics`와 $10^{-12}$ 상대오차로 대조한다. 두 구현이 독립적으로 같은 값을 주면 이후 표의 '닫힌 형식' 열을 믿을 수 있다.

In [ ]:
# helper (spkit에 없어서 여기서 정의): Erlang C 직접 구현, SLA 확률, 총비용
def erlang_c(lam, mu, c):
    a = lam / mu
    rho = a / c
    assert rho < 1, "unstable: rho >= 1"
    head = sum(a**k / math.factorial(k) for k in range(c))
    tail = a**c / (math.factorial(c) * (1 - rho))
    p_wait = tail / (head + tail)
    Lq = p_wait * rho / (1 - rho)
    return {"rho": rho, "p_wait": p_wait, "Lq": Lq, "L": Lq + a, "Wq": Lq / lam, "W": Lq / lam + 1 / mu}

def p_exceed_mmc(lam, mu, c, t):
    """정리 2: P(Wq > t) = C(c, a) exp(-(c mu - lam) t)."""
    return erlang_c(lam, mu, c)["p_wait"] * math.exp(-(c * mu - lam) * t)

def total_cost(c, lam=LAM, mu=MU, cs=COST_SERVER, cw=COST_WAIT):
    return cs * c + cw * lam * erlang_c(lam, mu, c)["Wq"]

C_RANGE = range(4, 10)
for c in C_RANGE:
    mine, ref = erlang_c(LAM, MU, c), mmc_metrics(LAM, MU, c)
    for key in mine:
        np.testing.assert_allclose(mine[key], ref[key], rtol=1e-12)
print("inline erlang_c == spkit.mmc_metrics (rtol 1e-12) for c = 4..9")

In [ ]:
lines = ["| c | ρ | C(c,a) = P(wait) | Lq | L | Wq (min) | E[Wq \\| Wq>0] (min) | P(Wq > 1 min) | TC (/h) |", "|---|---|---|---|---|---|---|---|---|"]
p_exceed_exact, tc_exact = {}, {}
for c in C_RANGE:
    m = mmc_metrics(LAM, MU, c)
    p_exceed_exact[c] = p_exceed_mmc(LAM, MU, c, T_SLA)
    tc_exact[c] = total_cost(c)
    lines.append(f"| {c} | {m['rho']:.4f} | {m['p_wait']:.6f} | {m['Lq']:.6f} | {m['L']:.6f} | {m['Wq']*60:.5f} "
                 f"| {60/(c*MU-LAM):.4f} | {p_exceed_exact[c]:.6f} | {tc_exact[c]:.2f} |")
c_star = min(c for c in C_RANGE if p_exceed_exact[c] <= ALPHA)
c_cost_opt = min(C_RANGE, key=tc_exact.get)
print(f"c* (SLA) = {c_star},  cost-only optimum = {c_cost_opt}")
Markdown("\n".join(lines))

In [ ]:
cs = np.array(list(C_RANGE))
fig, ax = plt.subplots()
ax.semilogy(cs, [p_exceed_exact[c] for c in cs], "o-", label="P(Wq > 1 min) = C exp(-(c mu - lambda)/60)")
ax.semilogy(cs, [mmc_metrics(LAM, MU, c)["p_wait"] for c in cs], "s--", label="Erlang C = P(Wq > 0)")
ax.axhline(ALPHA, color="red", ls=":", label=f"SLA alpha = {ALPHA}")
ax.axvline(c_star, color="gray", ls="--", alpha=0.6)
ax.annotate(f"c* = {c_star}", (c_star + 0.1, ALPHA * 1.6))
ax.set_xlabel("number of servers c"); ax.set_ylabel("probability (log)")
ax.set_title(f"SLA curve, lambda={LAM:g}/h, mu={MU:g}/h (a = {LAM/MU:.3f} Erlang)")
ax.legend(); plt.show()

$c = 5 \to 6$에서 $P(W_q > 1\text{분})$이 $0.234 \to 0.087$로 2.7배 떨어진다. 두 효과가 곱해지기 때문이다: $C(c, a)$ 자체가 줄고($0.327 \to 0.148$), 지수 꼬리의 율 $c\mu - \lambda$가 $20 \to 32$/h로 커져 $e^{-(c\mu-\lambda)/60}$이 $0.72 \to 0.59$로 줄어든다.

## 5. 단계 2 — 비용 곡선과 제약 최적화

$TC(c) = c_s c + c_w \lambda W_q(c)$. 리틀의 법칙으로 대기 비용 항은 $c_w L_q(c)$ — '평균 대기 인원 × 시간당 대기 비용'과 같다. 한 줄로 확인하고 비용 곡선을 그린다.

In [ ]:
for c in C_RANGE:
    m = mmc_metrics(LAM, MU, c)
    np.testing.assert_allclose(COST_WAIT * LAM * m["Wq"], COST_WAIT * m["Lq"], rtol=1e-12)   # 리틀: λ Wq = Lq
server_cost = COST_SERVER * cs
wait_cost = np.array([COST_WAIT * mmc_metrics(LAM, MU, c)["Lq"] for c in cs])
total = server_cost + wait_cost
shadow_price = tc_exact[c_star] - tc_exact[c_cost_opt]
print("TC(c):", {int(c): round(float(t), 2) for c, t in zip(cs, total)})
print(f"unconstrained optimum c = {c_cost_opt} ({tc_exact[c_cost_opt]:.2f}/h), "
      f"SLA-feasible optimum c = {c_star} ({tc_exact[c_star]:.2f}/h), shadow price of SLA = {shadow_price:.2f}/h")

fig, ax = plt.subplots()
ax.bar(cs, server_cost, color="tab:blue", label="server cost 30 c")
ax.bar(cs, wait_cost, bottom=server_cost, color="tab:orange", label="waiting cost 60 Lq")
ax.plot(cs, total, "k.-", label="total cost TC(c)")
ax.axvspan(cs[0] - 0.5, c_star - 0.5, color="red", alpha=0.08, label="SLA infeasible")
ax.set_xlabel("number of servers c"); ax.set_ylabel("cost per hour")
ax.set_title("Hourly cost vs number of servers"); ax.legend(); plt.show()

무제약 최적은 $c = 5$(약 189/h), 제약 최적은 $c = 6$(약 191/h). 차이 약 1.9/h가 SLA의 **그림자 가격**(shadow price) — SLA 한 줄이 시간당 이 정도 비용을 요구한다. 비용 곡선은 완만하지만 SLA 곡선은 절벽이라 두 답이 갈린다(함정 1).

## 6. 단계 3 — M/M/c를 두 시뮬레이터로 검증

### 6.1 CTMC 쪽 (03c): 절단 출생–사망 생성원과 Gillespie

$N(t)$는 출생률 $\lambda$, 사망률 $\min(n, 6)\mu$인 출생–사망 CTMC다. $n_{\max} = 60$에서 절단하면 잘리는 확률은 $\rho^{54} \cdot \pi_{6} < 10^{-14}$ 수준이어서 정상분포는 $10^{-10}$ 정밀도로 Erlang C와 일치해야 한다.

In [ ]:
C0, N_MAX = 6, 60
Q = birth_death_generator(birth=[LAM] * N_MAX, death=[min(k, C0) * MU for k in range(N_MAX + 1)])
assert is_generator(Q)
pi_lin = stationary_ctmc(Q)
pi_prod = birth_death_stationary([LAM] * N_MAX, [min(k, C0) * MU for k in range(N_MAX + 1)])
m6 = mmc_metrics(LAM, MU, C0)
np.testing.assert_allclose(pi_lin, pi_prod, atol=1e-12)
np.testing.assert_allclose(pi_lin[C0:].sum(), m6["p_wait"], atol=1e-10)     # P(N >= 6) = Erlang C
np.testing.assert_allclose(pi_lin @ np.arange(N_MAX + 1), m6["L"], atol=1e-10)  # E[N] = L
print(f"truncated birth-death (n_max={N_MAX}): P(N>=6) = {pi_lin[C0:].sum():.6f}, L = {pi_lin @ np.arange(N_MAX+1):.6f}, "
      f"tail mass pi[60] = {pi_lin[-1]:.1e}")

In [ ]:
t_grid, X = gillespie_paths(Q, x0=0, T=5.0, rng=rng, n_paths=N_PATHS, n_grid=101)
X_T = X[:, -1]                      # 시각 t = 5h의 상태: 경로마다 독립 (완화시간 ≈ 0.2h << 5h)
p_wait_gil = mc_proportion(X_T >= C0)
L_gil = mc_estimate(X_T)

times_long, states_long = gillespie(Q, 0, T_LONG, rng)
occ = time_average_occupation(times_long, states_long, T_LONG, N_MAX + 1)
print(f"Gillespie t=5h, {N_PATHS} paths: P(N>=6) = {p_wait_gil},  L = {L_gil}")
print(f"one long path T={T_LONG:g}h ({len(times_long)} jumps): time-average P(N>=6) = {occ[C0:].sum():.4f}, "
      f"L = {occ @ np.arange(N_MAX+1):.4f}  (no SE: one path)")

fig, ax = plt.subplots()
plot_paths(t_grid, X, ax=ax, n_show=5, step=True)
ax.axhline(C0, color="red", ls=":", label="c = 6 servers busy")
ax.set_ylabel("N(t) customers in system"); ax.set_xlabel("t (hours)")
ax.set_title("M/M/6 Gillespie sample paths from an empty system"); ax.legend(); plt.show()

### 6.2 고객 쪽: Kiefer–Wolfowitz 재귀

반복 축 $R$을 벡터화한다: `w`의 shape는 `(R, c)`, 고객 축만 루프. `gaps[:, k]`는 고객 $k$와 $k+1$의 도착 간격이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의): GI/G/c FCFS 대기시간 (Kiefer–Wolfowitz), 반복 축 벡터화
def mgc_waiting_times(gaps, services, c):
    """gaps, services: shape (R, n). 반환 W: shape (R, n), W[r, k] = 반복 r에서 고객 k의 대기시간."""
    R_, n = gaps.shape
    w = np.zeros((R_, c))                 # 정렬된 잔여 작업량 벡터 (각 반복마다)
    W = np.empty((R_, n))
    for k in range(n):
        W[:, k] = w[:, 0]                 # 가장 먼저 비는 서버까지 남은 시간 = FCFS 대기
        w[:, 0] += services[:, k]         # 그 서버에 내 서비스를 얹는다
        w -= gaps[:, k][:, None]          # 다음 도착까지 모든 서버의 작업량이 줄어든다
        np.maximum(w, 0.0, out=w)         # 0 아래로는 못 내려간다
        if c > 1:
            w.sort(axis=1)                # 순서가 바뀌었을 수 있으니 다시 정렬
    return W

def replication_stats(W, t_sla=T_SLA, warmup=WARMUP):
    """반복별 통계량 (각각 길이 R): 1분 초과 비율, 대기 비율, 평균 대기, 양의 대기의 평균."""
    Wc = W[:, warmup:]
    pos = Wc > 0
    return {"p_exceed": (Wc > t_sla).mean(1), "p_wait": pos.mean(1), "wq": Wc.mean(1),
            "cond": (Wc * pos).sum(1) / np.maximum(pos.sum(1), 1)}

In [ ]:
stats_mmc, W_mmc = {}, {}
for c in [5, 6, 7]:
    gaps = rng.exponential(1 / LAM, (R, N_CUST))
    services = rng.exponential(1 / MU, (R, N_CUST))
    W_mmc[c] = mgc_waiting_times(gaps, services, c)
    stats_mmc[c] = replication_stats(W_mmc[c])
    est = mc_estimate(stats_mmc[c]["p_exceed"])
    print(f"M/M/{c}: P(Wq > 1 min) = {est}   exact {p_exceed_exact[c]:.4f}   "
          f"P(wait) = {mc_estimate(stats_mmc[c]['p_wait']):.4f}   exact {mmc_metrics(LAM, MU, c)['p_wait']:.4f}")

SE는 **반복별 비율 200개**의 표준편차/$\sqrt{200}$이다. 고객 단위로 계산한 SE와 얼마나 다른지 확인해 보자 — 자기상관 때문에 고객 단위 SE는 훨씬 작게(잘못) 나온다(함정 2).

In [ ]:
Wc6 = W_mmc[6][:, WARMUP:]
se_rep = mc_estimate(stats_mmc[6]["wq"]).se
se_naive = Wc6.std(ddof=1) / np.sqrt(Wc6.size)          # 모든 고객을 i.i.d.로 취급한 (틀린) SE
print(f"M/M/6 mean Wq: replication SE = {se_rep*60:.4f} min, naive per-customer SE = {se_naive*60:.4f} min, "
      f"ratio = {se_rep/se_naive:.2f}")

waits6 = np.sort(Wc6.ravel() * 60)                        # 워밍업 이후 모든 대기 (분)
t_min = np.linspace(0, 6, 121)
surv_emp = 1 - np.searchsorted(waits6, t_min, side="right") / waits6.size
surv_exact = m6["p_wait"] * np.exp(-(C0 * MU - LAM) * t_min / 60)
fig, ax = plt.subplots()
ax.semilogy(t_min, surv_emp, label=f"simulation ({R} replications, pooled)")
ax.semilogy(t_min, surv_exact, "--", label="C(c,a) exp(-(c mu - lambda) t)")
ax.axvline(1.0, color="gray", ls=":"); ax.axhline(ALPHA, color="red", ls=":", label="SLA: t = 1 min, alpha = 0.10")
ax.set_xlabel("t (minutes)"); ax.set_ylabel("P(Wq > t)"); ax.set_ylim(1e-3, 1)
ax.set_title("M/M/6 waiting-time survival function"); ax.legend(); plt.show()

로그 눈금에서 직선 — 조건부 대기가 정확히 지수분포(정리 2)라는 뜻이다. 절편이 Erlang C, 기울기가 $-(c\mu - \lambda)$.

## 7. 단계 4 — M/G/1로 시뮬레이터를 비지수 서비스에서 검증 (Pollaczek–Khinchine)

같은 시뮬레이터에 서비스 표본기만 바꿔 끼운다. $\lambda = 9.6$/h, $\mu = 12$/h($\rho = 0.8$), $c = 1$에서 P–K는 D/M/H2에 대해 10/20/50분을 예측한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의): 평균 1/mu, 제곱변동계수 scv인 서비스시간 표본기
def h2_params(mu, scv=4.0):
    p1 = 0.5 * (1 + math.sqrt((scv - 1) / (scv + 1)))     # 균형 초지수: p1/mu1 = (1-p1)/mu2
    return p1, 2 * p1 * mu, 2 * (1 - p1) * mu

def service_sampler(kind, size, rng, mu=MU, scv=4.0):
    if kind == "exp":
        return rng.exponential(1 / mu, size)
    if kind == "det":
        return np.full(size, 1 / mu)
    if kind == "h2":
        p1, mu1, mu2 = h2_params(mu, scv)
        is_fast = rng.random(size) < p1
        return np.where(is_fast, rng.exponential(1 / mu1, size), rng.exponential(1 / mu2, size))
    if kind == "logn":
        s2 = math.log(1 + scv)
        return rng.lognormal(-math.log(mu) - s2 / 2, math.sqrt(s2), size)
    raise ValueError(kind)

SCV = {"det": 0.0, "exp": 1.0, "h2": 4.0, "logn": 4.0}
p1, mu1, mu2 = h2_params(MU)
print(f"H2: p1 = {p1:.5f}, mu1 = {mu1:.4f}, mu2 = {mu2:.4f} /h")
for kind in ["det", "exp", "h2", "logn"]:            # 표본기 검증: 평균 1/mu, SCV
    s = service_sampler(kind, 200_000, rng)
    print(f"  {kind:4s}: mean = {s.mean()*60:.3f} min (exact 5), SCV = {s.var()/s.mean()**2:.3f} (exact {SCV[kind]:g})")

In [ ]:
LAM1, C1 = 9.6, 1
rho1 = LAM1 / MU
def pk_wq(lam, mu, scv):
    rho = lam / mu
    return rho * (1 + scv) / (2 * mu * (1 - rho))      # 정리 4 (시간 단위)

wq_mg1_reps, wq_mg1_exact = {}, {}
for kind in ["det", "exp", "h2"]:
    gaps = rng.exponential(1 / LAM1, (R, N_CUST))
    W = mgc_waiting_times(gaps, service_sampler(kind, (R, N_CUST), rng), C1)
    wq_mg1_reps[kind] = replication_stats(W)["wq"] * 60
    wq_mg1_exact[kind] = pk_wq(LAM1, MU, SCV[kind]) * 60
    print(f"M/{kind}/1 (rho={rho1:.2f}): Wq = {mc_estimate(wq_mg1_reps[kind]):.3f} min   P-K {wq_mg1_exact[kind]:.2f} min")

fig, ax = plt.subplots()
grid = np.linspace(0, 4.5, 50)
ax.plot(grid, pk_wq(LAM1, MU, grid) * 60, "k--", label="Pollaczek-Khinchine rho(1+SCV)/(2 mu (1-rho))")
for kind in ["det", "exp", "h2"]:
    e = mc_estimate(wq_mg1_reps[kind])
    ax.errorbar(SCV[kind], e.mean, yerr=1.96 * e.se, fmt="o", capsize=4, label=f"M/{kind.upper()}/1 simulation (95% CI)")
ax.set_xlabel("service SCV  c_s^2"); ax.set_ylabel("mean queueing delay Wq (min)")
ax.set_title(f"M/G/1, lambda={LAM1}/h, mu={MU:g}/h (rho={rho1:.2f})"); ax.legend(); plt.show()

$\rho = 0.8$에서 대기 평균은 $c_s^2$에 **선형**이고 비율은 $1 : 2 : 5$. 결정적 서비스는 지수의 절반, $c_s^2 = 4$인 초지수는 2.5배다. 시뮬레이터가 비지수 서비스에서도 옳다는 것을 P–K가 보증해 주었으니, 이제 닫힌 형식이 없는 M/G/c로 간다.

## 8. 단계 5 — M/G/c 강건성: 같은 $c$, 다른 서비스 분포

In [ ]:
C_LIST = [5, 6, 7] if fast() else [5, 6, 7, 8]
KINDS = ["exp", "det", "h2"] if fast() else ["exp", "det", "h2", "logn"]
p_exceed_reps, cond_reps, wq_reps = {}, {}, {}
for kind in KINDS:
    for c in C_LIST:
        gaps = rng.exponential(1 / LAM, (R, N_CUST))
        W = mgc_waiting_times(gaps, service_sampler(kind, (R, N_CUST), rng), c)
        st = replication_stats(W)
        p_exceed_reps[kind, c] = st["p_exceed"]
        cond_reps[kind, c] = st["cond"] * 60
        wq_reps[kind, c] = st["wq"] * 60

cols = ["c", "Erlang C exact"] + [f"P(Wq>1min) {k}" for k in KINDS] + ["1/(c mu - lam) (min)"] + [f"E[Wq|Wq>0] {k} (min)" for k in KINDS]
lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
for c in C_LIST:
    cells = [str(c), f"{p_exceed_exact[c]:.4f}"] + [f"{mc_estimate(p_exceed_reps[k, c]):.4f}" for k in KINDS]
    cells += [f"{60/(c*MU-LAM):.3f}"] + [f"{mc_estimate(cond_reps[k, c]).mean:.3f}" for k in KINDS]
    lines.append("| " + " | ".join(cells) + " |")
Markdown("\n".join(lines))

In [ ]:
fig, ax = plt.subplots()
markers = {"exp": "o", "det": "s", "h2": "^", "logn": "v"}
for kind in KINDS:
    ests = [mc_estimate(p_exceed_reps[kind, c]) for c in C_LIST]
    ax.errorbar(C_LIST, [e.mean for e in ests], yerr=[1.96 * e.se for e in ests], fmt=markers[kind] + "-",
                capsize=4, label=f"{kind} service (SCV = {SCV[kind]:g})")
ax.semilogy(C_LIST, [p_exceed_exact[c] for c in C_LIST], "k:", label="Erlang C closed form (M/M/c)")
ax.axhline(ALPHA, color="red", ls="--", label="SLA alpha = 0.10")
ax.set_yscale("log"); ax.set_xticks(C_LIST); ax.set_xlabel("number of servers c"); ax.set_ylabel("P(Wq > 1 min)")
ax.set_title("Robustness of the SLA to the service-time distribution"); ax.legend(); plt.show()

e_det, e_h2 = mc_estimate(p_exceed_reps["det", 6]), mc_estimate(p_exceed_reps["h2", 6])
print(f"c = 6: deterministic {e_det}  |  H2 {e_h2}  |  Erlang C {p_exceed_exact[6]:.4f}")
print(f"c = 7: H2 {mc_estimate(p_exceed_reps['h2', 7])}")
assert e_det.mean < ALPHA < e_h2.mean + 4 * e_h2.se, "robustness conclusion: det has slack, H2 is at/above the SLA"
lo, hi = e_h2.mean - 1.96 * e_h2.se, e_h2.mean + 1.96 * e_h2.se       # 95% CI
verdict = "위반" if lo > ALPHA else ("경계(95% CI가 0.10에 걸림)" if hi >= ALPHA else "간신히 만족(4 SE 이내)")
print(f"결론: 결정적 서비스면 6대로 여유({e_det.mean:.3f} < 0.10), H2(c_s²=4)면 6대는 {verdict} → 7대 권장.")

**결론 셀.** 같은 $c = 6$에서 결정적 서비스는 $\approx 0.066$으로 SLA에 여유가 있고, $c_s^2 = 4$ 초지수는 $\approx 0.10$(참고 실행 200 × 18,000명: $0.1007 \pm 0.0007$; 이 노트북의 실행에서는 CI가 0.10에 걸칠 수 있다)으로 SLA를 아슬아슬하게 위반하거나 경계에 걸린다 → 7대가 필요하다($\approx 0.034$). 조건부 대기 $E[W_q \mid W_q > 0]$은 지수 서비스에서만 $1/(c\mu - \lambda) = 1.875$분이고, D에서는 더 짧고 H2에서는 훨씬 길다 — 정리 2가 지수 서비스 전용이라는 증거다. 근사 $W_q \approx W_q^{M/M/c}(1 + c_s^2)/2$(함정 4)와 위 표의 $\bar W_q$를 비교해 보면 M/G/c에서는 $c_s^2$만으로 대기가 결정되지 않는다는 것도 보인다.

## 9. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
P = predictions
rows = [
    {"name": "M/M/6: P(wait) = P(N >= 6), Gillespie states at t = 5h", "predicted": P["p_wait_cstar"], "estimate": p_wait_gil, "exact": m6["p_wait"]},
    {"name": "M/M/6: L = E[N], Gillespie states at t = 5h", "predicted": None, "estimate": L_gil, "exact": m6["L"]},
    {"name": "M/M/5: P(Wq > 1 min), KW simulator", "predicted": None, "estimate": mc_estimate(stats_mmc[5]["p_exceed"]), "exact": p_exceed_exact[5]},
    {"name": "M/M/6: P(Wq > 1 min)", "predicted": None, "estimate": mc_estimate(stats_mmc[6]["p_exceed"]), "exact": p_exceed_exact[6]},
    {"name": "M/M/7: P(Wq > 1 min)", "predicted": None, "estimate": mc_estimate(stats_mmc[7]["p_exceed"]), "exact": p_exceed_exact[7]},
    {"name": "M/M/6: P(Wq > 0), customer average (PASTA vs Erlang C)", "predicted": P["p_wait_cstar"], "estimate": mc_estimate(stats_mmc[6]["p_wait"]), "exact": m6["p_wait"]},
    {"name": "M/M/6: Wq (min)", "predicted": P["wq_cstar_min"], "estimate": mc_estimate(stats_mmc[6]["wq"] * 60), "exact": m6["Wq"] * 60},
    {"name": "M/M/6: E[Wq | Wq > 0] (min)", "predicted": P["cond_wait_cstar_min"], "estimate": mc_estimate(stats_mmc[6]["cond"] * 60), "exact": 60 / (C0 * MU - LAM)},
    {"name": "M/D/1 (lam=9.6, mu=12): Wq (min)", "predicted": None, "estimate": mc_estimate(wq_mg1_reps["det"]), "exact": wq_mg1_exact["det"]},
    {"name": "M/M/1 (lam=9.6, mu=12): Wq (min)", "predicted": None, "estimate": mc_estimate(wq_mg1_reps["exp"]), "exact": wq_mg1_exact["exp"]},
    {"name": "M/H2/1 (lam=9.6, mu=12, SCV=4): Wq (min)", "predicted": P["wq_mh21_min"], "estimate": mc_estimate(wq_mg1_reps["h2"]), "exact": wq_mg1_exact["h2"]},
    {"name": "M/D/6: P(Wq > 1 min) (no closed form)", "predicted": P["p_exceed_det_c6"], "estimate": e_det, "exact": None},
    {"name": "M/H2/6: P(Wq > 1 min) (no closed form)", "predicted": P["p_exceed_h2_c6"], "estimate": e_h2, "exact": None},
    {"name": "c* (SLA) [closed form only]", "predicted": P["c_star"], "estimate": float(c_star), "exact": float(c_star)},
    {"name": "cost-only optimum c [closed form only]", "predicted": P["c_cost_opt"], "estimate": float(c_cost_opt), "exact": float(c_cost_opt)},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **Gillespie P(N ≥ 6), L**: 절단 생성원이나 `gillespie_paths`가 틀렸거나, $t = 5$h가 정상 상태에 이르기에 짧은 것(완화시간 ≈ 0.2h이므로 아니다).
- **M/M/5,6,7 P(Wq > 1분)**: KW 재귀 구현(정렬·클리핑 순서)이 틀렸거나 워밍업이 부족한 것. 세 $c$에서 같은 방향으로 어긋나면 워밍업, 하나만 어긋나면 우연(4 SE 밖은 1만 번에 한 번).
- **P(Wq > 0) 고객평균 vs Erlang C**: 시간평균(Gillespie 행)은 맞는데 이 행만 어긋나면 PASTA가 깨진 것 — 도착이 푸아송이 아니라는 뜻(E3에서 일부러 깨 본다).
- **Wq, E[Wq | Wq > 0]**: 평균은 맞고 조건부만 어긋나면 조건부 대기가 지수가 아니라는 뜻(정리 2의 FCFS·지수 가정).
- **M/G/1 P–K**: 서비스 표본기의 평균·SCV가 틀린 것(위에서 표본기를 따로 검증했다). H2는 분산이 커서 SE가 가장 크다.
- **M/D/6, M/H2/6**: 닫힌 형식이 없어 판정은 '—'. 내 예측의 방향(D는 아래, H2는 위)만 확인한다.

In [ ]:
assert c_star == 6 and c_cost_opt == 5
assert_close(p_wait_gil, m6["p_wait"], k=4, name="P(wait) gillespie")
assert_close(L_gil, m6["L"], k=4, name="L gillespie")
for c in [5, 6, 7]:
    assert_close(mc_estimate(stats_mmc[c]["p_exceed"]), p_exceed_exact[c], k=4, name=f"P(Wq>1min) M/M/{c}")
assert_close(mc_estimate(stats_mmc[6]["p_wait"]), m6["p_wait"], k=4, name="P(Wq>0) M/M/6 (PASTA)")
assert_close(mc_estimate(stats_mmc[6]["wq"] * 60), m6["Wq"] * 60, k=4, name="Wq M/M/6")
assert_close(mc_estimate(stats_mmc[6]["cond"] * 60), 60 / (C0 * MU - LAM), k=4, name="E[Wq|Wq>0] M/M/6")
for kind in ["det", "exp", "h2"]:
    assert_close(mc_estimate(wq_mg1_reps[kind]), wq_mg1_exact[kind], k=4, name=f"M/{kind}/1 P-K")
print("all Monte Carlo checks passed (4 SE, replication SE)")

## 10. 단계 6 — 설계 메모와 대기실 없는 대안

In [ ]:
def erlang_b(a, c):
    """M/M/c/c 차단확률 (a^c/c!) / sum_{k<=c} a^k/k!."""
    terms = [a**k / math.factorial(k) for k in range(c + 1)]
    return terms[-1] / sum(terms)

def blocking_bd(lam, mu, c, K):
    """M/M/c/K (대기실 K-c석) 차단확률 = 절단 출생–사망 정상분포의 마지막 성분."""
    return birth_death_stationary([lam] * K, [min(k, c) * mu for k in range(K + 1)])[-1]

a = LAM / MU
for c in [5, 6]:
    np.testing.assert_allclose(blocking_bd(LAM, MU, c, c), erlang_b(a, c), rtol=1e-12)
block = {(c, K): blocking_bd(LAM, MU, c, K) for c, K in [(5, 5), (6, 6), (6, 11)]}
print("Erlang B / blocking:", {f"M/M/{c}/{K}": f"{v:.6f}" for (c, K), v in block.items()})

memo = f"""| 항목 | 값 | 근거 |
|---|---|---|
| 추천 서버 수 | **{c_star}대** (지수 서비스 기준), 서비스 분산이 크면 {c_star + 1}대 | 단계 1, 5 |
| SLA 지표 $P(W_q > 1\\text{{분}})$ | Erlang C {p_exceed_exact[c_star]:.4f}; 시뮬레이션 {mc_estimate(stats_mmc[c_star]['p_exceed']):.4f} | 정리 2, 단계 3 |
| 대기확률 / 평균 대기 | {m6['p_wait']:.4f} / {m6['Wq']*60:.3f}분 (기다리는 고객만 {60/(C0*MU-LAM):.3f}분) | 정리 1, 2 |
| 시간당 비용 | {tc_exact[c_star]:.2f} (비용 최적 {c_cost_opt}대는 {tc_exact[c_cost_opt]:.2f}이지만 SLA 위반 {p_exceed_exact[c_cost_opt]:.3f}) | 단계 2 |
| 민감도: 결정적 서비스 | {e_det.mean:.4f} ± {e_det.se:.4f} (여유) | 단계 5 |
| 민감도: 초지수 $c_s^2 = 4$ | {e_h2.mean:.4f} ± {e_h2.se:.4f} ({verdict}) → 7대에서 {mc_estimate(p_exceed_reps['h2', 7]).mean:.4f} | 단계 5 |
| 대안: 대기실 없음 (M/M/c/c) | 차단확률 B(5) = {block[5, 5]:.4f}, B(6) = {block[6, 6]:.4f} | Erlang B |
| 대안: 대기실 5석 (M/M/6/11) | 차단확률 {block[6, 11]:.6f} | 절단 출생–사망 |"""
Markdown(memo)

**대기 vs 손실, 어떤 지표로 설계할까.** 대기실이 무한한 M/M/c에서는 '얼마나 기다리는가'($W_q$의 꼬리)가 지표이고, 대기실이 없는 M/M/c/c에서는 '얼마나 돌려보내는가'(Erlang B 차단확률)가 지표다. 같은 6대라도 차단 시스템은 7.2%의 고객을 잃는 대신 아무도 기다리지 않고, 대기 시스템은 아무도 잃지 않는 대신 14.8%가 기다린다(전체 고객의 8.7%, 즉 기다리는 고객의 약 59%는 1분을 넘긴다). 대기실 5석만 두어도 차단은 0.35%로 떨어지므로, 실무에서는 '작은 대기실 + 대기 SLA'의 혼합이 흔하다. 어느 쪽이든 설계 변수는 하나($c$)지만 **잃는 것의 종류**(시간 vs 고객)가 다르므로, 비용 항 $c_w$를 무엇에 붙일지가 먼저 정해져야 한다.

## 11. 연습문제

### E1 계산

$\lambda = 20$/h, $\mu = 10$/h ($a = 2$)인 M/M/3에서 $\pi_0$, Erlang C, $L_q$, $W_q$(분), $P(W_q > 1\text{분})$을 **손으로** 계산하라. 그리고 $P(W_q > 1\text{분}) \le 0.10$을 만족하는 최소 $c$를 $c = 4, 5$를 손으로 시험해 찾아라. `mmc_metrics(20, 10, c)`로 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$c = 3$, $\rho = 2/3$: $\pi_0^{-1} = 1 + 2 + \frac{2^2}{2!} + \frac{2^3/3!}{1 - 2/3} = 1 + 2 + 2 + \frac{4/3}{1/3} = 9$ → $\pi_0 = 1/9$. $C = \frac{4/3}{1/3}\cdot\frac19 = \frac49 \approx 0.4444$. $L_q = C\rho/(1-\rho) = \frac49\cdot 2 = \frac89 \approx 0.8889$. $W_q = L_q/\lambda = \frac{8}{180}$h $= 2.667$분. $P(W_q > 1\text{분}) = \frac49 e^{-(30-20)/60} = 0.4444 \times 0.8465 = 0.3762$.

$c = 4$: $\pi_0^{-1} = 1 + 2 + 2 + \frac43 + \frac{2/3}{1 - 1/2} = 7.6667$, $\pi_0 = 0.13043$, $C = \frac{4/3}{1}\cdot 0.13043 = 0.17391$, $P = 0.17391\,e^{-20/60} = 0.1246$ (위반). $c = 5$: $\pi_0^{-1} = 7 + \frac{4/15}{1 - 2/5} = 7.4444$, $\pi_0 = 0.13433$, $C = 0.4444 \times 0.13433 = 0.05970$, $P = 0.0597\,e^{-30/60} = 0.0362$ (만족) → $c = 5$.

```python
for c in [3, 4, 5]:
    m = mmc_metrics(20, 10, c)
    print(c, f"C={m['p_wait']:.5f} Lq={m['Lq']:.5f} Wq={m['Wq']*60:.4f} min "
             f"P(Wq>1min)={m['p_wait']*math.exp(-(10*c-20)/60):.5f}")
# 3 C=0.44444 Lq=0.88889 Wq=2.6667 min P(Wq>1min)=0.37621
# 4 C=0.17391 Lq=0.17391 Wq=0.5217 min P(Wq>1min)=0.12461
# 5 C=0.05970 Lq=0.03980 Wq=0.1194 min P(Wq>1min)=0.03621
```

</details>

### E2 유도 후 시뮬레이션 검증

기하($1-\rho$)개의 i.i.d. Exp($c\mu$)의 합이 Exp($c\mu - \lambda$)임을 mgf(00c)로 유도하라. 그런 다음 단계 3의 M/M/6 Kiefer–Wolfowitz 시뮬레이션(`W_mmc[6]`)에서 워밍업 이후 **양의 대기시간만** 모아 히스토그램을 그리고 밀도 $32 e^{-32t}$(시간 단위)와 겹쳐라. 반복별 조건부 평균의 평균이 $1.875$분과 4 SE 이내인지 `assert_close`하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$N \sim$ 기하($1-\rho$) on $\{1, 2, \dots\}$ ($P(N = j+1) = (1-\rho)\rho^j$), $X_i \sim$ Exp($c\mu$) i.i.d., $T = \sum_{i=1}^N X_i$. 조건부 기댓값으로
$$E[e^{sT}] = \sum_{j \ge 0}(1-\rho)\rho^j\Big(\frac{c\mu}{c\mu - s}\Big)^{j+1} = \frac{(1-\rho)c\mu}{c\mu - s}\cdot\frac{1}{1 - \rho c\mu/(c\mu - s)} = \frac{(1-\rho)c\mu}{c\mu - s - \rho c\mu} = \frac{c\mu(1-\rho)}{c\mu(1-\rho) - s},$$
Exp($c\mu(1-\rho)$)의 mgf이고 $c\mu(1-\rho) = c\mu - \lambda$ ($s < c\mu - \lambda$에서 급수 수렴). $c = 6$: 율 $72 - 40 = 32$/h, 평균 $1/32$h $= 1.875$분.

```python
pos6 = Wc6[Wc6 > 0]                                    # 양의 대기 (시간 단위)
rate = C0 * MU - LAM                                   # 32/h
t = np.linspace(0, 0.3, 200)
ax = plot_hist_vs_pdf(pos6, t, rate * np.exp(-rate * t), bins=60, label_pdf="32 exp(-32 t)")
ax.set_xlabel("Wq | Wq > 0 (hours)"); ax.set_title("M/M/6 conditional waiting time"); plt.show()
cond6 = mc_estimate(stats_mmc[6]["cond"] * 60)
assert_close(cond6, 60 / rate, k=4, name="E[Wq|Wq>0]")
print(cond6, "vs", 60 / rate)   # 참고 실행(full): 1.882 ± 0.010 분
```

</details>

### E3 가정을 깨보기

(a) 도착이 푸아송이 아니라 **정확히 1.5분 간격**(D/M/6)이면 $P(W_q > 1\text{분})$은? 도착 간격이 Erlang-4(평균 1.5분; E4/M/6)이면? Erlang C의 0.087과 비교하고, 왜 PASTA가 필요한지 설명하라. (b) $\lambda = 75$/h, $c = 6$ ($\rho = 75/72 \approx 1.04$)에서 고객 5,000 / 10,000 / 20,000번째(FAST: 500 / 1,000 / 2,000번째)의 평균 대기(분)를 재고 선형 증가를 확인하라 — Erlang C 공식(`mmc_metrics`)은 무엇을 돌려주는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 참고 실행(full): D/M/6 $\approx 0.0115 \pm 0.0001$, E4/M/6 $\approx 0.0257 \pm 0.0002$ — 규칙적인 도착은 같은 평균 도착률에서도 대기를 크게 줄인다(M/M/6의 0.087 대비 1/7, 1/3). Erlang C는 **푸아송 도착 전용**이다: 비푸아송 도착에서는 '도착 고객이 보는 분포 ≠ 시간평균 분포'(PASTA 붕괴)이므로 $P(\text{wait}) \ne \sum_{n \ge c}\pi_n$이고, 애초에 $N(t)$가 출생–사망 과정도 아니다.

(b) 유체 근사: 고객당 순 증가 $1/(c\mu) - 1/\lambda = (1/72 - 1/75)$h $= 0.0333$분 → $n = 5{,}000 / 10{,}000 / 20{,}000$에서 약 $167 / 333 / 667$분. 시뮬레이션은 랜덤워크 요동($\sqrt{n}$ 규모) 때문에 유체 근사보다 크다: 참고 실행(full) 약 187 / 360 / 704분. FAST(500 / 1,000 / 2,000번째)에서는 약 27 / 47 / 84분 vs 유체 17 / 33 / 67분 — 초과분의 절대 크기(10–40분)는 비슷하지만 $n$이 작아 상대적으로 훨씬 커 보인다. 두 모드 모두 기울기(≈ 0.033–0.035분/고객)의 선형 증가는 확인된다. `mmc_metrics(75, 12, 6)`은 $\rho \ge 1$에서 `ValueError`를 낸다 — 정상분포가 없기 때문이다(직접 공식에 넣으면 $1/(1-\rho) < 0$이라 음수가 나온다).

```python
def p_exceed_from_gaps(gaps, c=6):
    W = mgc_waiting_times(gaps, rng.exponential(1 / MU, gaps.shape), c)
    return mc_estimate(replication_stats(W)["p_exceed"])
print("D/M/6 :", p_exceed_from_gaps(np.full((R, N_CUST), 1 / LAM)))
print("E4/M/6:", p_exceed_from_gaps(rng.gamma(4, 1 / (4 * LAM), (R, N_CUST))))
print("M/M/6 :", p_exceed_exact[6])

W_over = mgc_waiting_times(rng.exponential(1 / 75, (R, N_CUST)), rng.exponential(1 / MU, (R, N_CUST)), 6)
for n in [N_CUST // 4, N_CUST // 2, N_CUST - 1]:
    print(f"customer {n}: mean wait {W_over[:, n].mean()*60:.1f} min, fluid {n*(1/72 - 1/75)*60:.1f} min")
try:
    mmc_metrics(75, 12, 6)
except ValueError as err:
    print("mmc_metrics:", err)
```

</details>

### 스트레치 S1–S3 (선택)

**S1 (시간 변동 도착, 02c NHPP).** $\lambda(t) = 40 + 20\sin(\pi t/6)$ (12시간 주기, 피크 60/h). 30분 구간마다 정상 근사(SIPP: stationary independent period by period)로 Erlang C 인원 $c(t)$를 정하고, `nhpp_thinning`으로 도착을 만든 뒤 KW 재귀로 실제 $P(W_q > 1\text{분})$을 구간별로 측정하라.

<details><summary>풀이 개요</summary>

피크 $\lambda = 60$: $c = 7$에서 $P(W_q > 1\text{분}) = 0.2173$, $c = 8$에서 $0.0918$(만족), $c = 9$에서 $0.0362$. 시간대별 $c(t) \in \{5, \dots, 8\}$. 도착 시각을 `np.diff`로 간격 배열로 바꿔 KW에 넣는다. 서버 수가 시간에 따라 바뀌면 벡터 크기가 바뀌므로, 구간별로 재귀를 이어 붙이거나(마지막 $w$를 다음 구간의 초기값으로, 크기가 줄면 가장 큰 성분을 버리고 늘면 0을 덧붙임) 최대 $c$로 두고 일부 서버를 '가짜 작업'으로 막는 트릭을 쓴다. SIPP는 부하가 오르는 구간에서 대기를 과소평가한다(시스템이 정상 상태를 따라잡지 못한다).

</details>

**S2 (이탈, Erlang A).** 고객 인내심 $\sim$ Exp($\theta$), $\theta = 6$/h(평균 10분). 이벤트 기반 시뮬레이션(`heapq`)으로 M/M/6+M의 이탈 비율과 $P(W_q > 1\text{분})$을 추정하고 이탈이 없는 경우와 비교하라. 출생–사망 생성원(사망률 $\min(n, c)\mu + (n-c)^+\theta$)으로 정상분포를 정확히 구해 이탈 비율 $= \theta L_q/\lambda$를 검증하라.

<details><summary>풀이 개요</summary>

Erlang A는 출생–사망이므로 `birth_death_generator` / `birth_death_stationary`(절단 $n_{\max} = 80$)로 정확한 $\pi$가 나온다. 이탈률 $= \theta\sum_{n > c}(n-c)\pi_n/\lambda = \theta L_q/\lambda$ (대기 중인 각 고객이 율 $\theta$로 떠나므로 이탈 흐름은 $\theta L_q$, 이를 도착률로 나눈다). 시뮬레이션은 heap 기반 이벤트 루프(도착, 서비스 완료, 이탈 세 종류 이벤트)를 inline으로 작성한다; KW 재귀는 이탈을 표현하지 못한다(작업 보존이 깨진다).

</details>

**S3 (빠른 서버 1대 vs 느린 서버 2대, 03e).** $\lambda = 16$/h에서 (i) $c = 1$, $\mu = 24$ (ii) $c = 2$, $\mu = 12$. $W$, $W_q$, $P(W_q > 1\text{분})$을 비교하고, 서비스가 결정적일 때 결론이 바뀌는지 시뮬레이션하라.

<details><summary>풀이 개요</summary>

둘 다 $\rho = 2/3$. M/M/1: $W_q = \rho/(\mu - \lambda) = 0.0833$h $= 5$분, $W = 7.5$분, $P(W_q > 1\text{분}) = \frac23 e^{-8/60} = 0.583$. M/M/2: $C = 0.5333$, $L_q = 1.0667$, $W_q = 4$분, $W = 9$분, $P(W_q > 1\text{분}) = 0.5333\,e^{-8/60} = 0.467$. 느린 두 대가 **대기**는 짧지만 **체류**는 길다 — SLA가 $W_q$에 걸려 있으면 두 대, $W$에 걸려 있으면 한 대가 답이다. 결정적 서비스에서는 `service_sampler("det", ..., mu=24)` / `mu=12`로 두 경우를 시뮬레이션하면 된다.

</details>

## 12. 정리

1. M/M/c의 정상분포는 출생–사망 곱 형태에서 나오고, Erlang C $= \pi_0 a^c/(c!(1-\rho))$는 도착 고객이 기다릴 확률이다(PASTA). $a = 10/3$, $c = 6$: 0.1482.
2. 기다리는 고객의 대기는 Exp($c\mu - \lambda$)이므로 $P(W_q > t) = C e^{-(c\mu - \lambda)t}$; SLA $P(W_q > 1\text{분}) \le 0.10$은 $c = 5$(0.234)에서 깨지고 $c = 6$(0.087)에서 만족한다.
3. 비용 곡선의 최소는 $c = 5$(189.2/h)지만 SLA 실현가능 집합 안의 최소는 $c = 6$(191.1/h) — 제약 최적화의 답은 실현가능 집합 안에서 찾는다.
4. Kiefer–Wolfowitz 재귀로 만든 M/G/c 시뮬레이터는 M/M/c(Erlang C)와 M/G/1(P–K: 10/20/50분)을 4 SE 이내로 재현했고, 올바른 SE는 반복 실험으로만 나온다.
5. 서비스 변동성이 설계를 바꾼다: 결정적 서비스면 $c = 6$이 여유(0.066), $c_s^2 = 4$면 $c = 6$이 경계·위반(≈ 0.10) → $c = 7$. Erlang C는 $c_s^2 = 1$인 '중간' 답이다.

**trap 카드**
- Q: 서버를 5대에서 6대로 늘리면 시간당 비용이 189.2에서 191.1로 오르니 5대가 답 아닌가?
- A: 아니다. 5대는 $P(W_q > 1\text{분}) = 0.234$로 SLA(0.10)를 위반한다. 제약 최적화는 실현가능한 $c \ge 6$ 중에서 최소를 고른다 — 그리고 서비스 분산이 크면($c_s^2 = 4$) 6대도 약 0.10으로 아슬아슬하게 실패하므로 여유(7대)를 둘지 판단해야 한다.

**다음 노트북**: 04a — 리틀의 법칙과 '시간평균 = 고객평균'이 왜 성립하는지는 재생–보상 정리가 답한다. 모듈 04(재생 이론)에서 $L = \lambda W$와 $\pi(x) = 1/E_x[T_x]$를 증명한다.

log/progress.md 한 줄 템플릿:
`- [ ] C2 캡스톤 2: 대기행렬 설계 — 날짜: ____ / 예측 적중: __/8 / 단계 1-6 __ / E1 __ E2 __ E3 __ / 메모: ____`